In [1]:
from gurobipy import Model, GRB, quicksum
from pathlib import Path
import pandas as pd
import numpy as np
from pulp import *
import matplotlib.pyplot as plt
import os

# Excel-Datei auswerten
excel_file = Path(r"C:\1Masterarbeit\Szenario_1\Inputdaten_1.xlsx")
input_folder = excel_file.parent  

df = pd.read_excel(excel_file)

df_products = df.iloc[[0,1], 1:5].copy()
df_products.columns = ['db_i', 'b_i', 'e_i', 'v_i']

df_limits_raw = df.iloc[3:6, 1].copy()
df_limits_raw.name = 'Wert'

df_limits_params = df.iloc[3:6, 0].copy()
df_limits = pd.DataFrame({'Parameter': df_limits_params, 'Wert': df_limits_raw})

n_produkte = len(df_products)

db = df_products['db_i'].values.astype(float)
b = df_products['b_i'].values.astype(float)
e = df_products['e_i'].values.astype(float)
v = df_products['v_i'].values.astype(float)

BV_max = float(df_limits[df_limits['Parameter'] == 'BV_max']['Wert'].iloc[0])
E_max = float(df_limits[df_limits['Parameter'] == 'E_max']['Wert'].iloc[0])
W_max = float(df_limits[df_limits['Parameter'] == 'W_max']['Wert'].iloc[0])


#Optimierungsproblem
def solve_gurobi_optimization(scenario_name, include_all_constraints=False):
     
    model = Model(f"Produktionsprogrammplanung_{scenario_name}")
    model.Params.OutputFlag = 0  
    
    x = model.addVars(n_produkte, lb=0, vtype=GRB.CONTINUOUS, name="x")
    
    # Zielfunktion 
    model.setObjective(quicksum(db[i] * x[i] for i in range(n_produkte)), GRB.MAXIMIZE)
    
    # NB BV 
    model.addConstr(quicksum(b[i] * x[i] for i in range(n_produkte)) <= BV_max, name="NB_25_MSA")
    
    # NB Umwelt
    constr_names = {}
    if include_all_constraints:
        constr_names["NB_26_THG"] = model.addConstr(quicksum(e[i] * x[i] for i in range(n_produkte)) <= E_max, name="NB_26_THG")
        constr_names["NB_27_Wasser"] = model.addConstr(quicksum(v[i] * x[i] for i in range(n_produkte)) <= W_max, name="NB_27_Wasser")
    
    model.optimize()
    
    status = model.Status
    x_optimal = [x[i].X for i in range(n_produkte)]
    gewinn = model.ObjVal
    
    total_bv = sum(b[i] * x_optimal[i] for i in range(n_produkte))
    total_e = sum(e[i] * x_optimal[i] for i in range(n_produkte))
    total_v = sum(v[i] * x_optimal[i] for i in range(n_produkte))
    
    status_dict = {
    1: "LOADED", 2: "OPTIMAL", 3: "INFEASIBLE", 4: "INF_OR_UNBD", 
    5: "UNBOUNDED", 6: "CUTOFF", 7: "ITERATION_LIMIT", 8: "NODE_LIMIT",
    9: "TIME_LIMIT", 10: "SOLUTION_LIMIT", 11: "INTERRUPTED", 
    12: "ERR_NUMERIC", 13: "ERR_SUBOPTIMAL"
    }
    
    print(f"Status: {status} ({status_dict.get(status, 'UNKNOWN')})")
    print(f"Gewinn G: {gewinn:.2f}")
    print(f"x1: {x_optimal[0]:.4f}, x2: {x_optimal[1]:.4f}")
    

    #SA
    #reduzierte Kosten
    for i in range(n_produkte):
        rc = x[i].RC
        sa_obj_low = x[i].SAObjLow  
        sa_obj_up = x[i].SAObjUp   
        print(f"x_{i+1:<6} {x_optimal[i]:<12.4f} {rc:<10.4f} {sa_obj_low:<10.4f} {sa_obj_up:<10.4f}")
    
    # Schattenpreise
    for c in model.getConstrs():
        pi = c.Pi
        slack = c.Slack
        rhs = c.RHS
        sa_rhs_low = c.SARHSLow
        sa_rhs_up = c.SARHSUp
        print(f"{c.ConstrName:<15} {pi:<10.4f} {slack:<10.4f} {rhs:<10.0f} {sa_rhs_low:<10.2f} {sa_rhs_up:<10.2f}")
    #BV oder Nicht-BV
    basis_status = []
    for i in range(n_produkte):
        if x_optimal[i] > 1e-6:
            basis_status.append(f"x_{i+1} (Basis)")
        else:
            basis_status.append(f"x_{i+1} (Non-Basis, RC={x[i].RC:.4f})")

    
    sensitivity_data = {
        'Variable': [f'x_{i+1}' for i in range(n_produkte)],
        'Value': x_optimal,
        'ReducedCost': [x[i].RC for i in range(n_produkte)],
        'SAObjLow': [x[i].SAObjLow for i in range(n_produkte)], 
        'SAObjUp': [x[i].SAObjUp for i in range(n_produkte)],    
    }
    
    constr_data = []
    for c in model.getConstrs():
        constr_data.append({
            'Constraint': c.ConstrName,
            'ShadowPrice': c.Pi,
            'Slack': c.Slack,
            'RHS': c.RHS,
            'SARHSLow': c.SARHSLow,    
            'SARHSUp': c.SARHSUp,      
        })
    

    
    return {
        'scenario': scenario_name,
        'model': model, 
        'gewinn': gewinn,
        'x_optimal': x_optimal,
        'bv_total': total_bv,
        'e_total': total_e,
        'v_total': total_v,
        'sensitivity_vars': sensitivity_data,
        'sensitivity_constr': constr_data,
        'basis_status': basis_status,
    }

results_1A = solve_gurobi_optimization("1-A", False)
results_1H = solve_gurobi_optimization("1-H", True)

# Excel-Export
def export_results_to_excel(results_1A, results_1H, output_folder):
    output_file = output_folder / f"Ergebnisse.xlsx"
    
    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        summary_data = {
            'Szenario': ['1-A', '1-H'],
            'G*': [results_1A['gewinn'], results_1H['gewinn']],
            'x1*': [results_1A['x_optimal'][0], results_1H['x_optimal'][0]],
            'x2*': [results_1A['x_optimal'][1], results_1H['x_optimal'][1]],
            'BV Total': [results_1A['bv_total'], results_1H['bv_total']],
            'THG-Emissionen Total': [results_1A['e_total'], results_1H['e_total']],
            'Wasserverbrauch Total': [results_1A['v_total'], results_1H['v_total']],
        }
        pd.DataFrame(summary_data).to_excel(writer, sheet_name='Zusammenfassung', index=False)
        pd.DataFrame(results_1A['sensitivity_vars']).to_excel(writer, sheet_name='SA_Variablen_1A', index=False)
        pd.DataFrame(results_1A['sensitivity_constr']).to_excel(writer, sheet_name='SA_Constraints_1A', index=False)
        pd.DataFrame(results_1H['sensitivity_vars']).to_excel(writer, sheet_name='SA_Variablen_1H', index=False)
        pd.DataFrame(results_1H['sensitivity_constr']).to_excel(writer, sheet_name='SA_Constraints_1H', index=False)
        
    return output_file

output_file = export_results_to_excel(results_1A, results_1H, input_folder)





Set parameter Username
Set parameter LicenseID to value 2776645
Academic license - for non-commercial use only - expires 2027-02-08
Status: 2 (OPTIMAL)
Gewinn G: 2039.39
x1: 0.0000, x2: 18.5062
x_1      0.0000       -23.0936   -inf       122.3936  
x_2      18.5062      0.0000     89.4071    inf       
NB_25_MSA       1.5242     0.0000     1338       0.00       inf       
Status: 2 (OPTIMAL)
Gewinn G: 1842.21
x1: 8.5382, x2: 9.0232
x_1      8.5382       0.0000     77.1400    122.3936  
x_2      9.0232       0.0000     89.4071    141.8571  
NB_25_MSA       0.7464     0.0000     1338       1172.98    1720.71   
NB_26_THG       18.7456    0.0000     45         34.99      47.29     
NB_27_Wasser    0.0000     16.7854    210        193.21     inf       
